In [2]:
import astropy.units as u
import astropy.coordinates as apycoords
import pandas as pd
import numpy as np

In [3]:
def xyzuvw(ra, dec, distance, pmra, pmdec, radial_velocity):
    """
    Converts ICRS ra (deg), dec (deg), parallax (mas), pmra (mas/yr),
    pmdec (mas/yr), and radial velocity (km/s) to heliocentric
    Cartesian XYZ (pc) and UVW (km/s)

    Args:
        ra (array-like): Right ascension in degrees
        dec (array-like): Declination in degrees
        distance (array-like): Distance in parsecs
        pmra (array-like): Proper motion in right ascension (mas/yr)
        pmdec (array-like): Proper motion in declination (mas/yr)
        radial_velocity (array-like): Radial velocity in km/s

    Returns:
        numpy.ndarray : (n x 6) XYZUVW array
    """

    c = apycoords.SkyCoord(
        ra=ra * u.deg,
        dec=dec * u.deg,
        distance=distance * u.pc,
        pm_ra_cosdec=pmra * u.mas / u.yr,
        pm_dec=pmdec * u.mas / u.yr,
        radial_velocity=radial_velocity * u.km / u.s,
        frame="icrs",
    )

    cg = c.transform_to(apycoords.Galactic())
    cg.representation_type = "cartesian"

    xyz = cg.cartesian.xyz.value.T.reshape(-1, 3)
    uvw = np.vstack([cg.U.value, cg.V.value, cg.W.value]).T
    xyzuvw = np.concatenate([xyz, uvw], axis=1)

    return xyzuvw


def galcen_cyl(ra, dec, distance, pmra, pmdec, radial_velocity):
    """
    Converts ICRS ra (deg), dec (deg), parallax (mas), pmra (mas/yr),
    pmdec (mas/yr), and radial velocity (km/s) to Galactocentric cylindrical coordinates
    rho (kpc), phi (deg), z (pc), v_rho (km/s), v_phi (km/s), v_z (km/s)

    LEFT-HANDED

    Args:
        ra (array-like): Right ascension in degrees
        dec (array-like): Declination in degrees
        distance (array-like): Distance in parsecs
        pmra (array-like): Proper motion in right ascension (mas/yr)
        pmdec (array-like): Proper motion in declination (mas/yr)
        radial_velocity (array-like): Radial velocity in km/s

    Returns:
        numpy.ndarray: (n x 6) array of galactocentric cylindrical coordinates


    """

    c = apycoords.SkyCoord(
        ra=ra * u.deg,
        dec=dec * u.deg,
        distance=distance * u.pc,
        pm_ra_cosdec=pmra * u.mas / u.yr,
        pm_dec=pmdec * u.mas / u.yr,
        radial_velocity=radial_velocity * u.km / u.s,
        frame="icrs",
    )

    v_sun = apycoords.CartesianDifferential([11.1, 245.0, 7.25] * u.km / u.s)
    gc_frame = apycoords.Galactocentric(
        galcen_distance=8.25 * u.kpc, z_sun=20.8 * u.pc, galcen_v_sun=v_sun
    )

    gc = c.transform_to(gc_frame)
    gc.representation_type = "cylindrical"

    cyl_coord = np.vstack(
        [
            gc.rho.to(u.kpc).value,
            180 - gc.phi.degree,  # 180 - PHI ASTROPY CONVENTION
            gc.z.to(u.pc).value,
            gc.d_rho.to(u.km / u.s).value,
            -(gc.d_phi * gc.rho)
            .to(u.km / u.s, equivalencies=u.dimensionless_angles())
            .value,  # FLIPPED SIGN FROM ASTROPY CONVENTION
            gc.d_z.to(u.km / u.s).value,
        ]
    ).T

    return cyl_coord

In [4]:
hclu = pd.read_csv("/Volumes/travelpassport/litclusterdatabases/HR24/HR24_clusters.csv")
hmem = pd.read_hdf("/Volumes/travelpassport/litclusterdatabases/HR24/HR24_members.h5")


open_cond = (
    (hclu.dist16 < 500) & (hclu.CST > 5) & (hclu.Type == "o") & (hclu.RV.notna())
)
mg_cond = (hclu.dist16 < 500) & (hclu.CST > 10) & (hclu.Type == "m") & (hclu.RV.notna())
cond = open_cond | mg_cond

hclu = hclu.loc[cond].reset_index(drop=True).copy()

hmem = hmem.loc[hmem.Name.isin(hclu.Name)].reset_index(drop=True).copy()

In [5]:
median_rv = hmem.groupby("Name")["RV"].median()
hclu["median_member_RV"] = hclu["Name"].map(median_rv)

median_parallax = hmem.groupby("Name")["Plx"].median()
hclu["median_member_parallax"] = hclu["Name"].map(median_parallax)

In [6]:
df = hclu[
    [
        "Name",
        "RA_ICRS",
        "DE_ICRS",
        "GLON",
        "GLAT",
        "pmRA",
        "pmDE",
        "median_member_parallax",
        "median_member_RV",
    ]
]

df = df.rename(
    columns={
        "Name": "name",
        "RA_ICRS": "ra",
        "DE_ICRS": "dec",
        "GLON": "l",
        "GLAT": "b",
        "pmRA": "pmra",
        "pmDE": "pmdec",
    }
)

In [7]:
df[["x", "y", "z", "u", "v", "w"]] = xyzuvw(
    df.ra.values,
    df.dec.values,
    1000 / df.median_member_parallax.values,
    df.pmra.values,
    df.pmdec.values,
    df.median_member_RV.values,
)

df[["rho", "phi", "z_cyl", "v_rho", "v_phi", "v_z"]] = galcen_cyl(
    df.ra.values,
    df.dec.values,
    1000 / df.median_member_parallax.values,
    df.pmra.values,
    df.pmdec.values,
    df.median_member_RV.values,
)

In [8]:
df.to_csv("../data/clu_params.csv", index=False)